# Lab 03.5 – Thermal Alarm and Automatic Protection

## Objectives
- Monitor the Zynq die temperature continuously through XADC/IIO.
- Define NORMAL, WARNING, PROTECTION and SAFE thermal states.
- Generate a controlled CPU workload and stop it automatically when a software trip threshold is reached.
- Apply hysteresis so that the protection does not repeatedly switch around one threshold.
- Observe a latched alarm and the cooldown behavior after shutdown.

## Hardware
- Digilent Zybo legacy board
- PYNQ 3.0.1
- No external sensor or additional hardware is required.

## Important safety note
This laboratory implements a **software supervisory protection** at deliberately conservative temperatures. It does **not** attempt to trigger the Zynq hardware over-temperature shutdown. The hardware OT mechanism is a last-resort device-protection feature and is not used as a normal laboratory operating point.


## 1. Initialize XADC and LEDs

LD0 is used as a heartbeat, LD1 indicates WARNING, LD2 indicates active thermal protection, and LD3 is a latched alarm that remains set after a trip until the experiment ends.


In [ ]:
from pynq import Overlay
from pathlib import Path
import time
import os
import statistics
import multiprocessing as mp
import matplotlib.pyplot as plt

ol = Overlay("base.bit")

leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)
leds.write(0, 0b1111)

iio_root = Path("/sys/bus/iio/devices")
xadc_path = None

for dev in sorted(iio_root.glob("iio:device*")):
    name_file = dev / "name"
    if name_file.exists() and name_file.read_text().strip() == "xadc":
        xadc_path = dev
        break

if xadc_path is None:
    raise RuntimeError("XADC IIO device not found.")

def read_temperature_celsius():
    raw = float((xadc_path / "in_temp0_raw").read_text().strip())
    offset = float((xadc_path / "in_temp0_offset").read_text().strip())
    scale = float((xadc_path / "in_temp0_scale").read_text().strip())
    return (raw + offset) * scale / 1000.0

print("XADC and LED GPIO initialized.")
print(f"Current die temperature: {read_temperature_celsius():.2f} °C")


## 2. Define the controlled CPU workload

The workload uses at most two worker processes, matching the two ARM Cortex-A9 cores on the Zybo Zynq device. The stop event allows the monitor to terminate the workload immediately after a thermal trip.


In [ ]:
def cpu_worker(stop_event):
    x = 1.0
    while not stop_event.is_set():
        for _ in range(10000):
            x = (x * 1.000001 + 0.000001) % 1000.0

def start_cpu_load(workers=2):
    stop_event = mp.Event()
    processes = [mp.Process(target=cpu_worker, args=(stop_event,)) for _ in range(workers)]
    for p in processes:
        p.start()
    return stop_event, processes

def stop_cpu_load(stop_event, processes):
    if stop_event is None:
        return
    stop_event.set()
    for p in processes:
        p.join(timeout=2)
        if p.is_alive():
            p.terminate()
            p.join()

WORKERS = min(2, os.cpu_count() or 1)
print(f"CPU workers configured: {WORKERS}")


## 3. Establish the baseline and protection thresholds

Five temperature samples are averaged to obtain a startup baseline. The experiment then uses:

- WARNING = baseline + 3 °C
- TRIP = baseline + 5 °C, but never above 70 °C
- RESET = baseline + 2 °C

The lower RESET threshold introduces **hysteresis**. After a trip, the system is not considered thermally safe again until the temperature falls below RESET.

The workload is not started if the initial temperature is already 60 °C or higher.


In [ ]:
BASELINE_SAMPLES = 5
SAMPLE_INTERVAL = 1.0

WARNING_DELTA_C = 3.0
TRIP_DELTA_C = 5.0
RESET_DELTA_C = 2.0

MAX_START_TEMPERATURE_C = 60.0
ABSOLUTE_TRIP_C = 70.0
MAX_HEATING_TIME_S = 120
MAX_COOLDOWN_TIME_S = 60

baseline_samples = []
for i in range(BASELINE_SAMPLES):
    t = read_temperature_celsius()
    baseline_samples.append(t)
    print(f"Baseline sample {i + 1}: {t:.2f} °C")
    if i < BASELINE_SAMPLES - 1:
        time.sleep(SAMPLE_INTERVAL)

baseline_temperature = statistics.mean(baseline_samples)

if baseline_temperature >= MAX_START_TEMPERATURE_C:
    raise RuntimeError(
        f"Baseline temperature {baseline_temperature:.2f} °C is too high to start the heating experiment."
    )

warning_threshold = baseline_temperature + WARNING_DELTA_C
trip_threshold = min(baseline_temperature + TRIP_DELTA_C, ABSOLUTE_TRIP_C)
reset_threshold = baseline_temperature + RESET_DELTA_C

if warning_threshold >= trip_threshold:
    warning_threshold = trip_threshold - 1.0

print(f"\nBaseline temperature : {baseline_temperature:.2f} °C")
print(f"Warning threshold    : {warning_threshold:.2f} °C")
print(f"Trip threshold       : {trip_threshold:.2f} °C")
print(f"Reset threshold      : {reset_threshold:.2f} °C")


## 4. Run the automatic thermal-protection experiment

The CPU workload starts automatically. The monitor samples once per second.

- Below WARNING: state = `NORMAL`
- At or above WARNING: state = `WARNING`
- At or above TRIP: state = `PROTECTION`; the CPU workload is stopped immediately and the alarm is latched
- After shutdown, cooling continues to be monitored
- At or below RESET: state = `SAFE`

If the trip threshold is not reached within two minutes, the workload is stopped by timeout rather than being allowed to run indefinitely.


In [ ]:
times = []
temperatures = []
states = []

heartbeat = False
trip_latched = False
protection_active = False
trip_time = None

stop_event = None
processes = []
load_running = False

experiment_start = time.monotonic()

def update_leds(temp, heartbeat, protection_active, trip_latched):
    warning = temp >= warning_threshold
    pattern = 0
    pattern |= int(heartbeat) << 0
    pattern |= int(warning) << 1
    pattern |= int(protection_active) << 2
    pattern |= int(trip_latched) << 3
    leds.write(pattern, 0b1111)

def record_sample(state, protection_active, trip_latched):
    global heartbeat
    now = time.monotonic()
    temp = read_temperature_celsius()
    heartbeat = not heartbeat
    update_leds(temp, heartbeat, protection_active, trip_latched)
    elapsed = now - experiment_start
    times.append(elapsed)
    temperatures.append(temp)
    states.append(state)
    print(f"{state:<11} t={elapsed:6.1f} s   T={temp:6.2f} °C")
    return temp

try:
    stop_event, processes = start_cpu_load(WORKERS)
    load_running = True
    print("\nCPU workload started.\n")

    heat_start = time.monotonic()
    sample_index = 0

    while True:
        target = heat_start + sample_index * SAMPLE_INTERVAL
        delay = target - time.monotonic()
        if delay > 0:
            time.sleep(delay)

        temp = read_temperature_celsius()

        if temp >= trip_threshold:
            trip_latched = True
            protection_active = True
            trip_time = time.monotonic() - experiment_start

            stop_cpu_load(stop_event, processes)
            load_running = False

            heartbeat = not heartbeat
            update_leds(temp, heartbeat, protection_active, trip_latched)
            elapsed = time.monotonic() - experiment_start
            times.append(elapsed)
            temperatures.append(temp)
            states.append("PROTECTION")
            print(f"PROTECTION  t={elapsed:6.1f} s   T={temp:6.2f} °C")
            print("*** THERMAL TRIP: CPU workload stopped automatically. ***")
            break

        state = "WARNING" if temp >= warning_threshold else "NORMAL"
        heartbeat = not heartbeat
        update_leds(temp, heartbeat, protection_active, trip_latched)
        elapsed = time.monotonic() - experiment_start
        times.append(elapsed)
        temperatures.append(temp)
        states.append(state)
        print(f"{state:<11} t={elapsed:6.1f} s   T={temp:6.2f} °C")

        if time.monotonic() - heat_start >= MAX_HEATING_TIME_S:
            stop_cpu_load(stop_event, processes)
            load_running = False
            print("*** TIMEOUT: trip threshold was not reached; CPU workload stopped. ***")
            break

        sample_index += 1

    cooldown_start = time.monotonic()
    sample_index = 1

    while time.monotonic() - cooldown_start <= MAX_COOLDOWN_TIME_S:
        target = cooldown_start + sample_index * SAMPLE_INTERVAL
        delay = target - time.monotonic()
        if delay > 0:
            time.sleep(delay)

        temp = read_temperature_celsius()

        if trip_latched and temp <= reset_threshold:
            protection_active = False
            state = "SAFE"
        else:
            state = "COOLDOWN"

        heartbeat = not heartbeat
        update_leds(temp, heartbeat, protection_active, trip_latched)
        elapsed = time.monotonic() - experiment_start
        times.append(elapsed)
        temperatures.append(temp)
        states.append(state)
        print(f"{state:<11} t={elapsed:6.1f} s   T={temp:6.2f} °C")

        if state == "SAFE":
            print("Thermal reset threshold reached. System is thermally safe again.")
            break

        sample_index += 1

finally:
    if load_running:
        stop_cpu_load(stop_event, processes)
    leds.write(0, 0b1111)
    print("\nExperiment finished; CPU workload stopped and all LEDs are OFF.")


## 5. Plot the protection sequence


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(times, temperatures, marker="o", label="Die temperature")
plt.axhline(warning_threshold, linestyle="--", label="Warning threshold")
plt.axhline(trip_threshold, linestyle="--", label="Trip threshold")
plt.axhline(reset_threshold, linestyle="--", label="Reset threshold")

if trip_time is not None:
    plt.axvline(trip_time, linestyle="--", label="Automatic shutdown")

plt.xlabel("Time (s)")
plt.ylabel("Temperature (°C)")
plt.title("Thermal Alarm and Automatic Protection")
plt.grid(True)
plt.legend()
plt.show()


## 6. Results


In [ ]:
if temperatures:
    print(f"Initial recorded temperature: {temperatures[0]:.2f} °C")
    print(f"Maximum recorded temperature: {max(temperatures):.2f} °C")
    print(f"Final recorded temperature:   {temperatures[-1]:.2f} °C")

if trip_latched:
    print(f"Thermal protection activated at t = {trip_time:.1f} s.")
else:
    print("Thermal trip threshold was not reached during the allowed heating interval.")


## Questions
1. Why is the TRIP threshold deliberately much lower than the Zynq hardware over-temperature protection level?
2. What is the purpose of the separate WARNING state?
3. Why is RESET lower than TRIP? What problem would occur if both thresholds were identical?
4. Why is the trip alarm latched even after the die cools below the warning threshold?
5. Why is a maximum heating time required even though a temperature trip already exists?
6. What is the difference between this software protection and the XADC hardware OT protection?
7. In a real embedded product, which actions besides stopping a CPU workload could a thermal supervisor perform?
